In [ ]:
# We need specific versions for 4-bit loading on Colab
!pip install -q transformers accelerate bitsandbytes

from google.colab import drive
# Mount Drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 14.5 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import pandas as pd
import json
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline


# Paths
BASE_PATH = '/content/drive/MyDrive/imdb_data/ekko_artifacts'
INPUT_PATH = f'{BASE_PATH}/vip_list.csv'
OUTPUT_PATH = f'{BASE_PATH}/vip_llama_ai_tags.json'

print(f"Libraries installed. Torch version: {torch.__version__}")

Libraries installed. Torch version: 2.9.0+cu126


In [ ]:
# Configure 4-bit Quantization (To fit in free GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load Model & Tokenizer
# Using NousResearch mirror to avoid needing a HuggingFace Token
MODEL_ID = "NousResearch/Meta-Llama-3-8B-Instruct"

print(f"Loading {MODEL_ID}... (This takes ~2 mins)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Create Generation Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=64, # Keep it short to save time
    temperature=0.7,
    top_p=0.9
)

# Termination token (stops the model from rambling)
terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("Llama 3 Loaded & Ready.")

Loading NousResearch/Meta-Llama-3-8B-Instruct... (This takes ~2 mins)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Device set to use cuda:0


Llama 3 Loaded & Ready.


In [ ]:
def generate_tags(title, genres):
    messages = [
        {
            "role": "system",
            "content": "You are an expert in Movies and TV Shows. Provide 3 short, engaging 'Vibe Tags' (1-2 words each) and a (1-sentence or 15 words max) plot summary. \nSTRICT FORMAT: Tag1, Tag2, Tag3 | Summary\nNO INTRO OR OUTRO."
        },
        {
            "role": "user",
            "content": f"Title: {title} (Genres: {genres})"
        }
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Increased tokens slightly to allow for longer summaries if needed
    outputs = pipe(
        prompt,
        max_new_tokens=75,
        eos_token_id=terminators,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    result = outputs[0]["generated_text"][len(prompt):].strip()
    return result

# Test on tv-show
print("Testing on 'How I Met Your Mother':")
print(generate_tags("How I Met Your Mother", "Comedy, Romance"))

# Test it on one movie
print("Testing on 'The Matrix'...")
print(generate_tags("The Matrix", "Action, Sci-Fi"))

Testing on 'How I Met Your Mother':
Friendship, Nostalgia | The show follows Ted Mosby as he recounts to his son the story of how he met his children's mother in the 2000s.
Testing on 'The Matrix'...
Virtual Reality, Mindbending | In a dystopian future, Neo discovers he's living in a simulated reality created by machines.


In [ ]:
# Load VIP List
vip_df = pd.read_csv(INPUT_PATH)
print(f"Loaded {len(vip_df)} VIP Movie or TV Show.")

# Check for existing progress
results = {}
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, 'r') as f:
        results = json.load(f)
    print(f"Found existing progress: {len(results)} Movie or TV Show done.")

# Start Generation
print("Starting Generation... (Stop this cell anytime, progress is saved)")

count = 0
for index, row in vip_df.iterrows():
    tconst = str(row['tconst'])

    # Skip if already done
    if tconst in results:
        continue

    title = row['primaryTitle']
    # Extract genres from text_blob or use a placeholder if you prefer
    # (Assuming text_blob has genres, or we just pass the title for speed)

    try:
        # Generate
        ai_output = generate_tags(title, "Movie or TV Show")

        # Save to dict
        results[tconst] = {
            "title": title,
            "ai_insight": ai_output
        }

        count += 1

        # Print progress every 10 items
        if count % 10 == 0:
            print(f"Processed {len(results)}/{len(vip_df)}: {title} -> {ai_output}")

        # Save to file every 50 items (Checkpoint)
        if count % 50 == 0:
            with open(OUTPUT_PATH, 'w') as f:
                json.dump(results, f, indent=4)

    except Exception as e:
        print(f"Error on {title}: {e}")
        continue

# Final Save
with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=4)

print("VIP AI Tags Processing Complete!")

Loaded 2000 VIP Movie or TV Show.
Found existing progress: 1999 Movie or TV Show done.
Starting Generation... (Stop this cell anytime, progress is saved)
VIP AI Tags Processing Complete!


In [ ]:
import json
import os

# Define Path (Same as before)
BASE_PATH = '/content/drive/MyDrive/imdb_data/ekko_artifacts'
OUTPUT_PATH = f'{BASE_PATH}/vip_llama_ai_tags.json'

# Load the current data
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, 'r') as f:
        results = json.load(f)

    print(f"Loaded {len(results)} items.")

    # Find entries with fewer than 8 words
    # A good entry looks like: "Dark, Gritty, Fast | A man fights for his life." (12+ words)
    # A bad entry looks like: "Action, Fast" (2 words)

    bad_keys = []
    for tconst, data in results.items():
        text = data['ai_insight']
        word_count = len(text.split())

        if word_count < 12:
            bad_keys.append(tconst)
            print(f"🗑️ Marking for deletion ({word_count} words): {data['title']} -> {text}")

    # Delete them
    for tconst in bad_keys:
        del results[tconst]

    # Save back to disk
    with open(OUTPUT_PATH, 'w') as f:
        json.dump(results, f, indent=4)

    print(f"Purge Complete. Removed {len(bad_keys)} short entries.")
    print(f"Remaining valid items: {len(results)}")
    print("Now re-run the Generation Loop cell to fix them!")

else:
    print("No file found to clean.")

Loaded 2000 items.
Purge Complete. Removed 0 short entries.
Remaining valid items: 2000
Now re-run the Generation Loop cell to fix them!
